# Lorenz-96 system

Paper figures, drawn from the saved results. The state can be negative, so there is no non-negativity correction and no hAT-ROM.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../../../"))

import numpy as np

from src import plots as P
from src.run import Experiment, Results
from src.plots import (load_tables, move_corner_label, place_captions, set_font_sizes,
                       set_partition_titles, stack_captions, thin_tick_labels)
from matplotlib.ticker import NullFormatter

from src.fom.data import SUBSTEPS
from src.rom import ATROM
# After the imports: src.plots switches matplotlib to Agg.
%matplotlib inline

CASE = "lorenz96"
OUT_DIR = "."
SWEEP_DIR = "./sweep"
FIG_DIR = "../../../figures/lorenz96"
POSITIVITY = "off"

## Results

In [ ]:
sweep, rank, hatrom = load_tables(OUT_DIR, POSITIVITY)
taus = sorted(np.unique(sweep["test"]["tau_split"]))
etas = sorted(np.unique(sweep["test"]["eta"]), reverse=True)
qoi = [c.split("|", 1)[1] for c in rank["test"].dtype.names if c.startswith("l2-in-time|")]
n_eval = {s: int(np.max(sweep[s]["n_tracers"])) for s in sweep}
# Rank 100 keeps every mode.
rank = {s: r[r["residual_energy"] > 1e-15] for s, r in rank.items()}

## Errors

In [ ]:
fig_errors, axes = P.sweep_panel_splits(
    sweep["train"], rank["train"], sweep["test"], rank["test"], qoi,
    n_eval_train=n_eval["train"], n_eval_test=n_eval["test"],
    show_tols=tuple(taus), twin_tol=min(taus),
    hatrom_train=hatrom["train"], hatrom_test=hatrom["test"],
    ylim=(5e-5, 4e-1), sharex=False, row_gap=0.05, figsize=(20, 11.5), legend_ncol=6, legend_y=0.95,
)
set_font_sizes(fig_errors)
thin_tick_labels(fig_errors, [axes[0][1], axes[1][1]], {0, 1, 3, 5, 7})
for row in (0, 1):
    axes[row][1].set_ylim(5e-5, 3e-1)
    move_corner_label(axes[row][1])
    # The global error spans less than a decade: label the ticks by hand.
    axes[row][0].set_yticks([0.2, 0.3, 0.4, 0.5])
    axes[row][0].set_yticklabels(["0.2", "0.3", "0.4", "0.5"])
    axes[row][0].yaxis.set_minor_formatter(NullFormatter())

## Clusters and ranks

In [ ]:
# Below eta = 1e-6 every model has the 16 initial clusters, so the cluster bars stop there.
fig_partition, axes_p = P.sweep_partition_panel(
    sweep["test"], SWEEP_DIR, show_tols=tuple(taus), positivity=POSITIVITY, eval_split="test",
    max_eta_exp=6, bar_headroom=1.1, figsize=(15, 6), fom=False,
)
set_font_sizes(fig_partition)
set_partition_titles(fig_partition, axes_p)
thin_tick_labels(fig_partition, [axes_p[1]], {0, 2, 4, 6, 7})

## AT-ROM error distributions

In [ ]:
violin_taus = [0.25, 0.02, 0.005]
fig_violin, axes_v = P.violin_panel(
    SWEEP_DIR, violin_taus, qoi, etas=etas, positivity=POSITIVITY, eval_split="test",
    figsize=(5 * len(violin_taus) + 2, 5 * len(qoi)),
)
# 1e-8 and 1e-10 sit next to each other: drop the 1e-8 label.
for ax in axes_v[-1]:
    thin_tick_labels(fig_violin, [ax], {0, 1, 2, 4})

## QoI trajectories for one test forcing

FOM, AT-ROM (tau = 0.005, eta = 1e-2) and a global ROM with 91 modes, for the test forcing nearest 2.45.

In [ ]:
TAU, ETA, GLOBAL_RANK = 0.005, 1e-2, 91

exp = Experiment(OUT_DIR, "test", n_workers=os.cpu_count(), eta=ETA)
cfg, ctx = exp.cfg, exp.ctx

# The test forcing nearest 2.45. Each test case is stored as its forcing at every interval end.
_, forcings, _ = exp.split_files
i = np.argmin([abs(f[0, 0] - 2.45) for f in forcings])
F = forcings[i][0, 0]

# FOM solution, and the interval end times at which the ROMs are sampled.
_, x, _ = exp.trajectories([i])
x_fom = x[0]
t_ends = ctx.dt_hydro * np.arange(ctx.n_steps + 1)

# AT-ROM, and a global ROM fitted on all training snapshots.
atrom = exp.load_atrom(TAU, ETA)
global_rom = ATROM.fit_global(exp.train, ctx.net, cfg, ctx.cluster_indices, ctx.param_names, GLOBAL_RANK, cfg.n_workers)

opts = dict(dt_hydro=ctx.dt_hydro, atol=cfg.atol, rtol=cfg.rtol, t_eval=t_ends, positivity=False, method=cfg.ode_method)
# The last output is None unless the solve failed.
_, x_atrom, clusters, atrom_failure = atrom.solve_tracer(forcings[i], x_fom[:, 0], **opts)
_, x_global, _, global_failure = global_rom.solve_tracer(forcings[i], x_fom[:, 0], **opts)

mean_rank = np.mean([atrom.roms[c].basis.rank for c in clusters])
print(f"F = {F}: AT-ROM visits {len(set(clusters))} of {atrom.cluster.n_clusters} clusters, mean rank {mean_rank:.2f}")

# Serif fonts, as in the other paper figures.
P.use_paper_style()
fig_traj, _ = P.trajectory_panel(
    t_ends,
    {"reference": x_fom[:, ::SUBSTEPS],                        # FOM at the interval ends
     "clustered": x_atrom if atrom_failure is None else None,
     "global": x_global if global_failure is None else None},
    clusters, list(ctx.qoi_indices), figsize=(20, 10))
set_font_sizes(fig_traj, label=28, tick=22, xtick=22, legend=24)
fig_traj.tight_layout(rect=(0, 0, 1, 0.88))
fig_traj.text(0.5, 0.905, rf"$F = {F:.2f}$, $\tau_\mathrm{{split}} = {TAU:g}$, $\eta = 10^{{{np.log10(ETA):.0f}}}$",
              ha="center", va="center", fontsize=22)

## Save

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
fig_errors.savefig(f"{FIG_DIR}/panel_errors.pdf", dpi=200, bbox_inches="tight")
fig_partition.savefig(f"{FIG_DIR}/panel_partition.pdf", dpi=200, bbox_inches="tight")
fig_violin.savefig(f"{FIG_DIR}/violin_atrom.pdf", bbox_inches="tight")
fig_traj.savefig(f"{FIG_DIR}/trajectory.pdf", dpi=200, bbox_inches="tight")